# Notebook for plotting contingency tables
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path

import pandas as pd
import plotnine as p9

from utils.plotting_utils import make_plot_contingency_table
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
method_1 = "gmm"
method_2 = "kmeans"

# Load data
data_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
assert data_full.meta is not None
assert data_full.na is not None

proba_and_labels_1 = pd.read_parquet(Path(f"outputs/{method_1}_clustering/cluster_pred_discov.parquet"))
proba_and_labels_2 = pd.read_parquet(Path(f"outputs/{method_2}_clustering/cluster_pred_discov.parquet"))

# Set outputs folder
output_folder = Path("outputs/contingency_tables")
output_folder.mkdir(exist_ok=True)

# Set predicted probability and labels
proba_1 = proba_and_labels_1.drop("label", axis="columns")
proba_2 = proba_and_labels_2.drop("label", axis="columns")
labels_1 = proba_and_labels_1["label"]
labels_2 = proba_and_labels_2["label"]

# Set sample weights
sample_weight = data_full.data["wichtn"]

In [ ]:
# Show entailment frequencies using contingency tables (rows add to 1)
plot_cont_table1 = make_plot_contingency_table(
    labels_1, labels_2, name1=f"{method_1} clustering", name2=f"{method_2} clustering", sample_weight=sample_weight
)
plot_cont_table2 = make_plot_contingency_table(
    labels_2, labels_1, name1=f"{method_2} clustering", name2=f"{method_1} clustering", sample_weight=sample_weight
)

save_plot(plot_cont_table1, save_path=output_folder / f"plot_cont_table_{method_1}_vs_{method_2}.png", dpi=150)
save_plot(plot_cont_table2, save_path=output_folder / f"plot_cont_table_{method_2}_vs_{method_1}.png", dpi=150)

# Combine the two contingency table plots
plot_cont_table_all = (plot_cont_table1 + plot_cont_table2) & p9.theme(figure_size=(12, 5))

plot_cont_table_all.show()

save_plot(
    plot_cont_table_all,
    save_path=output_folder / f"plot_cont_table_{method_1}_vs_{method_2}_and_{method_2}_vs_{method_1}.png",
    dpi=150,
)